## Split Original Real-world Data into Train and Test Sets

In [2]:
# Stratified Data Split for KPD Phishing Dataset
# Split original 10K dataset into 2K balanced train and 2K balanced test sets
# Enhanced with comprehensive missing data handling

import pandas as pd
import numpy as np
import gzip
from pathlib import Path
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("🔄 Enhanced Dataset Stratified Split with Missing Data Handling")
print("=" * 70)

# =============================================================================
# CONFIGURATION
# =============================================================================

# File paths
INPUT_FILE = "../raw/five_email_phishing.csv.gz"
OUTPUT_TRAIN = "../raw/five_email_phishing_train.csv.gz"
OUTPUT_TEST = "../raw/five_email_phishing_test.csv.gz"

# Split configuration
TRAIN_SIZE = 2000
TEST_SIZE = 2000
RANDOM_STATE = 42  # For reproducibility

# Missing data handling configuration
DROP_MISSING = True  # Whether to drop rows with missing data
MISSING_THRESHOLD = 0.0  # Drop rows with > this fraction of missing values (0.0 = any missing)

print(f"📂 Input file: {INPUT_FILE}")
print(f"📁 Output train: {OUTPUT_TRAIN}")
print(f"📁 Output test: {OUTPUT_TEST}")
print(f"🎯 Target sizes: {TRAIN_SIZE} train, {TEST_SIZE} test")
print(f"🧹 Missing data handling: {'Drop rows with ANY missing values' if DROP_MISSING else 'Keep all rows'}")

# =============================================================================
# 1. LOAD ORIGINAL DATA
# =============================================================================

def load_data(file_path):
    """Load compressed CSV data."""
    print(f"\n📥 Loading data from: {file_path}")
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            df = pd.read_csv(f)
        
        print(f"✅ Data loaded successfully!")
        print(f"   Shape: {df.shape}")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        raise

# Load the original dataset
df_original = load_data(INPUT_FILE)

# =============================================================================
# 2. MISSING DATA ANALYSIS AND CLEANING
# =============================================================================

def analyze_missing_data(df):
    """Comprehensive missing data analysis."""
    print(f"\n🔍 Missing Data Analysis:")
    print("-" * 40)
    
    # Overall missing data stats
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    missing_percentage = (missing_cells / total_cells) * 100
    
    print(f"📊 Overall Statistics:")
    print(f"   Total cells: {total_cells:,}")
    print(f"   Missing cells: {missing_cells:,}")
    print(f"   Missing percentage: {missing_percentage:.2f}%")
    
    if missing_cells == 0:
        print(f"✅ No missing values found!")
        return df, 0
    
    # Per-column missing data analysis
    print(f"\n📋 Missing Data by Column:")
    missing_by_column = df.isnull().sum()
    missing_columns = missing_by_column[missing_by_column > 0]
    
    if len(missing_columns) > 0:
        print(f"   Columns with missing data:")
        for col, count in missing_columns.items():
            percentage = (count / len(df)) * 100
            print(f"      {col}: {count:,} ({percentage:.2f}%)")
    
    # Per-row missing data analysis
    print(f"\n📝 Missing Data by Row:")
    missing_per_row = df.isnull().sum(axis=1)
    rows_with_missing = (missing_per_row > 0).sum()
    rows_percentage = (rows_with_missing / len(df)) * 100
    
    print(f"   Rows with missing data: {rows_with_missing:,} ({rows_percentage:.2f}%)")
    
    if rows_with_missing > 0:
        # Distribution of missing values per row
        missing_distribution = missing_per_row.value_counts().sort_index()
        print(f"   Distribution of missing values per row:")
        for missing_count, row_count in missing_distribution.items():
            if missing_count > 0:
                print(f"      {missing_count} missing values: {row_count:,} rows")
    
    return df, rows_with_missing

def clean_missing_data(df, drop_missing=True, threshold=0.0):
    """Clean missing data based on configuration."""
    print(f"\n🧹 Missing Data Cleaning:")
    print("-" * 35)
    
    original_shape = df.shape
    
    if not drop_missing:
        print(f"   ⚠️  Keeping all rows (missing data not cleaned)")
        return df, 0
    
    # Count rows with missing data before cleaning
    rows_with_missing_before = (df.isnull().sum(axis=1) > 0).sum()
    
    if threshold == 0.0:
        # Drop rows with ANY missing values
        print(f"   🎯 Dropping rows with ANY missing values...")
        df_cleaned = df.dropna(how='any')
        method_description = "any missing values"
    else:
        # Drop rows with missing values above threshold
        print(f"   🎯 Dropping rows with >{threshold*100:.1f}% missing values...")
        missing_threshold_count = int(threshold * df.shape[1])
        df_cleaned = df.dropna(thresh=df.shape[1] - missing_threshold_count)
        method_description = f">{threshold*100:.1f}% missing values"
    
    # Reset index after dropping rows
    df_cleaned = df_cleaned.reset_index(drop=True)
    
    rows_dropped = original_shape[0] - df_cleaned.shape[0]
    percentage_dropped = (rows_dropped / original_shape[0]) * 100
    
    print(f"   📊 Cleaning Results:")
    print(f"      Original rows: {original_shape[0]:,}")
    print(f"      Rows with missing data: {rows_with_missing_before:,}")
    print(f"      Rows dropped: {rows_dropped:,}")
    print(f"      Rows remaining: {df_cleaned.shape[0]:,}")
    print(f"      Percentage dropped: {percentage_dropped:.2f}%")
    print(f"      Method: Drop rows with {method_description}")
    
    # Verify no missing data remains (if threshold was 0.0)
    if threshold == 0.0:
        remaining_missing = df_cleaned.isnull().sum().sum()
        if remaining_missing == 0:
            print(f"   ✅ All missing data successfully removed!")
        else:
            print(f"   ⚠️  WARNING: {remaining_missing} missing values still remain!")
    
    return df_cleaned, rows_dropped

# Analyze missing data in original dataset
df_analyzed, missing_rows = analyze_missing_data(df_original)

# Clean missing data if configured
df_clean, rows_dropped = clean_missing_data(df_original, DROP_MISSING, MISSING_THRESHOLD)

# =============================================================================
# 3. DATA EXPLORATION (POST-CLEANING)
# =============================================================================

print(f"\n📊 Dataset Analysis (After Cleaning):")
print("-" * 45)

# Basic info
print(f"Total samples: {len(df_clean):,}")
if rows_dropped > 0:
    print(f"Samples after cleaning: {len(df_clean):,} (dropped {rows_dropped:,})")

# Final missing values check
final_missing = df_clean.isnull().sum().sum()
if final_missing > 0:
    print(f"⚠️  Remaining missing values: {final_missing}")
    print(f"   Per column:")
    missing_final = df_clean.isnull().sum()
    for col, count in missing_final[missing_final > 0].items():
        print(f"      {col}: {count}")
else:
    print(f"✅ No missing values in cleaned dataset")

# Label distribution
print(f"\n🏷️  Label Distribution (Cleaned Data):")
if 'label' in df_clean.columns:
    label_counts = df_clean['label'].value_counts().sort_index()
    print(f"Benign (0): {label_counts[0]:,} ({label_counts[0]/len(df_clean)*100:.1f}%)")
    print(f"Malicious (1): {label_counts[1]:,} ({label_counts[1]/len(df_clean)*100:.1f}%)")
else:
    print(f"⚠️  No 'label' column found in dataset")
    print(f"Available columns: {list(df_clean.columns)}")

# Check if we have enough samples for desired split
total_needed = TRAIN_SIZE + TEST_SIZE
print(f"\n🔍 Feasibility Check (After Cleaning):")
print(f"Total needed: {total_needed:,}")
print(f"Available: {len(df_clean):,}")

if len(df_clean) < total_needed:
    print(f"⚠️  WARNING: Not enough data after cleaning! Need {total_needed:,}, have {len(df_clean):,}")
    # Adjust sizes based on available data
    available_samples = len(df_clean)
    max_per_set = available_samples // 2
    TRAIN_SIZE = min(TRAIN_SIZE, max_per_set)
    TEST_SIZE = min(TEST_SIZE, max_per_set)
    print(f"   Auto-adjusting to: {TRAIN_SIZE} train, {TEST_SIZE} test")
else:
    print(f"✅ Sufficient data available after cleaning")

# Check per-class availability
if 'label' in df_clean.columns:
    samples_per_class = max(TRAIN_SIZE, TEST_SIZE) // 2  # For balanced split
    min_class_size = label_counts.min()
    
    print(f"Samples needed per class: {samples_per_class:,}")
    print(f"Minimum class size: {min_class_size:,}")
    
    if min_class_size < samples_per_class:
        print(f"⚠️  WARNING: Not enough samples in minority class after cleaning!")
        print(f"   Adjusting to use maximum available: {min_class_size:,} per class")
        max_balanced_size = min_class_size * 2
        TRAIN_SIZE = min(TRAIN_SIZE, max_balanced_size)
        TEST_SIZE = min(TEST_SIZE, max_balanced_size)
        print(f"   Final sizes: {TRAIN_SIZE} train, {TEST_SIZE} test")

# =============================================================================
# 4. STRATIFIED SAMPLING
# =============================================================================

def stratified_split_balanced(df, train_size, test_size, label_col='label', random_state=42):
    """
    Create balanced train and test sets with stratified sampling.
    Each set will have equal numbers of each class.
    """
    print(f"\n🎯 Performing Stratified Balanced Split:")
    print(f"   Train size: {train_size} (balanced)")
    print(f"   Test size: {test_size} (balanced)")
    
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in dataset")
    
    # Separate by class
    benign_df = df[df[label_col] == 0]
    malicious_df = df[df[label_col] == 1]
    
    print(f"   Available - Benign: {len(benign_df):,}, Malicious: {len(malicious_df):,}")
    
    # Calculate samples needed per class for each set
    train_per_class = train_size // 2
    test_per_class = test_size // 2
    
    print(f"   Target per class - Train: {train_per_class}, Test: {test_per_class}")
    
    # Check availability
    total_per_class_needed = train_per_class + test_per_class
    
    if len(benign_df) < total_per_class_needed or len(malicious_df) < total_per_class_needed:
        raise ValueError(f"Not enough samples in one or both classes. Need {total_per_class_needed} per class.")
    
    # Sample from each class
    np.random.seed(random_state)
    
    # Benign samples
    benign_sampled = benign_df.sample(n=total_per_class_needed, random_state=random_state)
    benign_train = benign_sampled.iloc[:train_per_class]
    benign_test = benign_sampled.iloc[train_per_class:train_per_class + test_per_class]
    
    # Malicious samples  
    malicious_sampled = malicious_df.sample(n=total_per_class_needed, random_state=random_state)
    malicious_train = malicious_sampled.iloc[:train_per_class]
    malicious_test = malicious_sampled.iloc[train_per_class:train_per_class + test_per_class]
    
    # Combine and shuffle
    train_df = pd.concat([benign_train, malicious_train], ignore_index=True)
    test_df = pd.concat([benign_test, malicious_test], ignore_index=True)
    
    # Shuffle both sets
    train_df = train_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Verify results
    print(f"\n✅ Split Results:")
    print(f"   Train set: {len(train_df)} samples")
    train_counts = train_df[label_col].value_counts().sort_index()
    print(f"      Benign: {train_counts[0]}, Malicious: {train_counts[1]}")
    
    print(f"   Test set: {len(test_df)} samples") 
    test_counts = test_df[label_col].value_counts().sort_index()
    print(f"      Benign: {test_counts[0]}, Malicious: {test_counts[1]}")
    
    return train_df, test_df

# Perform the split
train_df, test_df = stratified_split_balanced(
    df_clean, 
    TRAIN_SIZE, 
    TEST_SIZE, 
    label_col='label', 
    random_state=RANDOM_STATE
)

# =============================================================================
# 5. VALIDATION
# =============================================================================

print(f"\n🔍 Data Validation:")
print("-" * 30)

# Check no overlap between train and test
# Note: Since we're using stratified sampling, we need to check the original indices
train_original_indices = set(train_df.index)
test_original_indices = set(test_df.index)
overlap = train_original_indices.intersection(test_original_indices)

if len(overlap) > 0:
    print(f"⚠️  WARNING: {len(overlap)} overlapping indices found!")
else:
    print(f"✅ No overlap between train and test sets")

# Verify no missing data in final sets
train_missing = train_df.isnull().sum().sum()
test_missing = test_df.isnull().sum().sum()

print(f"Missing values check:")
print(f"   Train set missing values: {train_missing}")
print(f"   Test set missing values: {test_missing}")

if train_missing == 0 and test_missing == 0:
    print(f"✅ No missing values in train or test sets")
else:
    print(f"⚠️  WARNING: Missing values detected in final datasets!")

# Verify label distributions
print(f"\n📊 Final Label Distributions:")

print(f"Training set:")
train_label_counts = train_df['label'].value_counts().sort_index()
for label, count in train_label_counts.items():
    label_name = "Benign" if label == 0 else "Malicious"
    print(f"   {label_name} ({label}): {count} ({count/len(train_df)*100:.1f}%)")

print(f"Test set:")
test_label_counts = test_df['label'].value_counts().sort_index()
for label, count in test_label_counts.items():
    label_name = "Benign" if label == 0 else "Malicious"
    print(f"   {label_name} ({label}): {count} ({count/len(test_df)*100:.1f}%)")

# Check data integrity
print(f"\n🔍 Data Integrity Check:")
print(f"Expected columns: {list(df_clean.columns)}")
print(f"Train columns match: {list(train_df.columns) == list(df_clean.columns)}")
print(f"Test columns match: {list(test_df.columns) == list(df_clean.columns)}")

# Verify data types are preserved
print(f"Data types preserved:")
types_match_train = (train_df.dtypes == df_clean.dtypes).all()
types_match_test = (test_df.dtypes == df_clean.dtypes).all()
print(f"   Train data types match: {types_match_train}")
print(f"   Test data types match: {types_match_test}")

# =============================================================================
# 6. SAVE DATA
# =============================================================================

def save_compressed_csv(df, file_path, description):
    """Save DataFrame to compressed CSV."""
    print(f"\n💾 Saving {description}...")
    print(f"   Path: {file_path}")
    print(f"   Samples: {len(df):,}")
    print(f"   Missing values: {df.isnull().sum().sum()}")
    
    try:
        # Create directory if it doesn't exist
        Path(file_path).parent.mkdir(parents=True, exist_ok=True)
        
        # Save compressed
        with gzip.open(file_path, 'wt', encoding='utf-8') as f:
            df.to_csv(f, index=False)
        
        # Verify file was created
        if Path(file_path).exists():
            file_size = Path(file_path).stat().st_size / 1024**2  # MB
            print(f"   ✅ Saved successfully! ({file_size:.2f} MB)")
        else:
            print(f"   ❌ Error: File not created")
            
    except Exception as e:
        print(f"   ❌ Error saving: {e}")
        raise

# Save training data
save_compressed_csv(train_df, OUTPUT_TRAIN, "training data")

# Save test data  
save_compressed_csv(test_df, OUTPUT_TEST, "test data")

# =============================================================================
# 7. SUMMARY
# =============================================================================

print(f"\n" + "="*70)
print(f"🎉 ENHANCED STRATIFIED DATA SPLIT COMPLETED")
print(f"="*70)

print(f"\n📊 Summary:")
print(f"   Original dataset: {len(df_original):,} samples")
if rows_dropped > 0:
    print(f"   After cleaning: {len(df_clean):,} samples ({rows_dropped:,} dropped)")
print(f"   Training set: {len(train_df):,} samples (balanced)")
print(f"   Test set: {len(test_df):,} samples (balanced)")
print(f"   Total extracted: {len(train_df) + len(test_df):,} samples")

print(f"\n🧹 Data Cleaning Summary:")
if DROP_MISSING:
    print(f"   Missing data handling: ENABLED")
    print(f"   Threshold: Drop rows with {('ANY' if MISSING_THRESHOLD == 0.0 else f'>{MISSING_THRESHOLD*100:.1f}%')} missing values")
    print(f"   Rows dropped: {rows_dropped:,}")
    print(f"   Missing values in final data: {train_df.isnull().sum().sum() + test_df.isnull().sum().sum()}")
else:
    print(f"   Missing data handling: DISABLED")
    print(f"   All rows preserved")

print(f"\n📁 Output Files:")
print(f"   🚂 Training: {OUTPUT_TRAIN}")
print(f"   🧪 Testing: {OUTPUT_TEST}")

print(f"\n🎯 Class Distribution (Both Sets):")
print(f"   Benign: 50% (balanced)")
print(f"   Malicious: 50% (balanced)")

print(f"\n✅ Ready for machine learning analysis!")
print(f"   Random state used: {RANDOM_STATE} (for reproducibility)")
print(f"   No data leakage: Train and test sets are completely separate")
print(f"   Data quality: Clean datasets with no missing values")

print(f"\n" + "="*70)

🔄 Enhanced Dataset Stratified Split with Missing Data Handling
📂 Input file: ../raw/five_email_phishing.csv.gz
📁 Output train: ../raw/five_email_phishing_train.csv.gz
📁 Output test: ../raw/five_email_phishing_test.csv.gz
🎯 Target sizes: 2000 train, 2000 test
🧹 Missing data handling: Drop rows with ANY missing values

📥 Loading data from: ../raw/five_email_phishing.csv.gz
✅ Data loaded successfully!
   Shape: (131346, 4)
   Columns: ['subject', 'body', 'label', 'source']
   Memory usage: 257.84 MB

🔍 Missing Data Analysis:
----------------------------------------
📊 Overall Statistics:
   Total cells: 525,384
   Missing cells: 754
   Missing percentage: 0.14%

📋 Missing Data by Column:
   Columns with missing data:
      subject: 753 (0.57%)
      body: 1 (0.00%)

📝 Missing Data by Row:
   Rows with missing data: 754 (0.57%)
   Distribution of missing values per row:
      1 missing values: 754 rows

🧹 Missing Data Cleaning:
-----------------------------------
   🎯 Dropping rows with ANY